# 04 Features: one row per municipality-year, with next year's label
Reads the `curated_*` and `clean_*` tables and writes `features_muni_year`, the table the model trains on.

- **Label (decision 12):** next year's opinion is **non-reliable** (adverse, disclaimer or outstanding) = 1, or **reliable** (unqualified, unqualified with findings, qualified) = 0.
- **No leakage:** every feature uses only what was known by the end of year *t*. The label comes from year *t+1*.
- **All 25 candidate features are built (decision 13).** No manual selection: regularisation in the model decides how much each one counts.
- Financial features are filled only where `fin_usable` is true (2019+, not flagged broken). Elsewhere they're left empty (decisions 5 and 10).

The last cell runs checks: every check must print PASS.

In [ ]:
from pyspark.sql import functions as F, Window  # Spark's column functions (F) and window tools (for looking at earlier and later years)

NON_RELIABLE = ["adverse", "disclaimer", "outstanding"]  # decision 12: these opinions mean the statements can't be relied on

panel = spark.table("curated_muni_year_panel")  # the curated municipality-year panel from notebook 03
inc = spark.table("clean_incexp").where(F.col("amount_type") == "AUDA")  # audited income and expenditure line items
fpos = spark.table("clean_financial_position").where(F.col("amount_type") == "AUDA")  # audited balance sheet line items
debt = spark.table("clean_aged_debtor").where((F.col("amount_type") == "AUDA") & (F.col("item_code") == "2000"))  # audited debtors, the "total by income source" row

by_muni = Window.partitionBy("demarcation_code").orderBy("year")  # a window: each municipality's rows, in year order
last_3 = by_muni.rowsBetween(-2, 0)  # this year and the two before it
so_far = by_muni.rowsBetween(Window.unboundedPreceding, 0)  # every year up to and including this one

## 1. Label and audit-history features (A1 to A6)

In [ ]:
hist = (panel
    .withColumn("non_reliable", F.col("opinion_code").isin(NON_RELIABLE).cast("int"))  # 1 if this year's opinion is non-reliable, else 0
    .withColumn("next_year", F.lead("year").over(by_muni))  # the year of this municipality's next opinion
    .withColumn("next_non_reliable", F.lead("non_reliable").over(by_muni))  # whether that next opinion is non-reliable
    # the label: only valid when the next opinion really is for the following year (no gap)
    .withColumn("label", F.when(F.col("next_year") == F.col("year") + 1, F.col("next_non_reliable")))
    .withColumn("a1_opinion_rank", F.col("opinion_rank"))  # A1: severity 1 (clean) to 5 (disclaimer); empty for outstanding
    .withColumn("a2_non_reliable_now", F.col("non_reliable"))  # A2: is this year already non-reliable?
    .withColumn("a3_non_reliable_last3", F.sum("non_reliable").over(last_3))  # A3: non-reliable opinions in the last 3 years (0 to 3)
    .withColumn("last_clean_year", F.max(F.when(F.col("is_clean"), F.col("year"))).over(so_far))  # the most recent year with a clean audit, if any
    # A4: years since the last clean audit; if never clean, years since the data starts (2011), so the number still grows
    .withColumn("a4_years_since_clean", F.coalesce(F.col("year") - F.col("last_clean_year"), F.col("year") - F.lit(2010)))
    .withColumn("a5_ever_disclaimer", F.max((F.col("opinion_code") == "disclaimer").cast("int")).over(so_far))  # A5: 1 if any disclaimer so far
    .withColumn("a6_opinion_trend", F.col("opinion_rank") - F.lag("opinion_rank").over(by_muni)))  # A6: change in severity from last year (+ = worse)

## 2. Financial building blocks per municipality-year

In [ ]:
def total(df, lo, hi, name):
    """Add up line items with codes from lo to hi into one number per municipality-year."""
    return (df.where(F.col("item_code").between(lo, hi))  # keep only the items in that code range
              .groupBy("demarcation_code", "year")  # one row per municipality-year
              .agg(F.sum("amount").alias(name)))  # add them up under the given name

def item(df, code, name):
    """Pick out one line item as its own column."""
    return (df.where(F.col("item_code") == code)  # keep only that item
              .groupBy("demarcation_code", "year")  # one row per municipality-year
              .agg(F.sum("amount").alias(name)))  # its amount, under the given name

KEY = ["demarcation_code", "year"]  # the columns that identify one municipality-year
blocks = [
    total(inc, "0300", "2500", "revenue"),  # operating revenue: service charges, rates, grants and so on (excludes one-off gains)
    total(inc, "3100", "4300", "expenditure"),  # total expenditure: all expense items
    total(inc, "3100", "3200", "employee_costs"),  # staff plus councillor pay
    item(inc, "3500", "debt_impairment"),  # revenue written off as uncollectable
    item(inc, "3600", "depreciation"),  # depreciation and amortisation (not a cash cost)
    item(inc, "3800", "contracted_services"),  # work paid to outside contractors
    item(inc, "2200", "operational_grants"),  # transfers and subsidies for operations
    total(fpos, "0120", "0180", "current_assets"),  # cash, receivables, inventory and other current assets
    total(fpos, "0350", "0420", "current_liabilities"),  # overdraft, payables, provisions and other current liabilities
    item(fpos, "0120", "cash"),  # cash and cash equivalents
    debt.select(*KEY, F.col("total_amount").alias("debtors_total"),  # total money owed to the municipality
                (F.col("l30_amount") + F.col("l60_amount") + F.col("l90_amount")).alias("debtors_0_90")),  # the part owed for 90 days or less
]
fin = blocks[0]  # start with the first block...
for b in blocks[1:]:  # ...and attach each of the others
    fin = fin.join(b, KEY, "full")  # "full" keeps a municipality-year even if it only appears in one block

## 3. Ratio features (C, D, E, F, G)

In [ ]:
def ratio(num, den):
    """Safe division: empty when the bottom number is missing or not positive."""
    return F.when(F.col(den) > 0, F.col(num) / F.col(den))  # avoids dividing by zero or by a negative total

feat = (hist.join(fin, KEY, "left")  # attach the financial building blocks to the history table
    .withColumn("usable", F.coalesce(F.col("fin_usable"), F.lit(False)))  # decisions 5 and 10: only use financials that are usable
    # B: reporting discipline. True means something is missing or broken.
    .withColumn("b1_missing_income_stmt", (~F.col("has_audited_incexp")).cast("int"))  # B1: no audited income statement (empty before 2018)
    .withColumn("b2_missing_balance_sheet", (~F.col("has_audited_fin_position")).cast("int"))  # B2: no audited balance sheet (empty before 2018)
    .withColumn("b3_missing_debtors", (~F.col("has_audited_debtors")).cast("int"))  # B3: no audited debtors (empty before 2020)
    .withColumn("b4_fin_flagged_broken", F.coalesce(F.col("fin_error"), F.lit(False)).cast("int"))  # B4: totals flagged broken in the outlier stage
    # C: irregular, fruitless and unauthorised expenditure, as a share of total expenditure (decision 11: scale, don't cap)
    .withColumn("c1_irregular_pct", F.when(F.col("usable"), ratio("irregular_exp", "expenditure")))  # C1
    .withColumn("c2_fruitless_pct", F.when(F.col("usable"), ratio("fruitless_exp", "expenditure")))  # C2
    .withColumn("c3_unauthorised_pct", F.when(F.col("usable"), ratio("unauthorised_exp", "expenditure")))  # C3
    .withColumn("log_irregular", F.log1p("irregular_exp"))  # log(1 + irregular), so zero stays zero and billions don't dominate
    .withColumn("c4_irregular_trend", F.col("log_irregular") - F.lag("log_irregular").over(by_muni))  # C4: change from last year on the log scale
    # D: liquidity and going concern
    .withColumn("d1_current_ratio", F.when(F.col("usable"), ratio("current_assets", "current_liabilities")))  # D1: current assets / current liabilities
    .withColumn("monthly_cash_cost", (F.col("expenditure") - F.coalesce("depreciation", F.lit(0)) - F.coalesce("debt_impairment", F.lit(0))) / 12)  # average monthly spend, excluding non-cash costs
    .withColumn("d2_cash_coverage_months", F.when(F.col("usable"), ratio("cash", "monthly_cash_cost")))  # D2: months the cash would last
    .withColumn("d3_operating_margin", F.when(F.col("usable"), (F.col("revenue") - F.col("expenditure")) / F.when(F.col("revenue") > 0, F.col("revenue"))))  # D3: surplus or deficit as a share of revenue
    # E: revenue and debt collection
    .withColumn("e1_debtors_over_90_pct", F.when(F.col("debtors_total") > 0, 1 - F.col("debtors_0_90") / F.col("debtors_total")))  # E1: share of debtors older than 90 days
    .withColumn("e2_debt_impairment_pct", F.when(F.col("usable"), ratio("debt_impairment", "revenue")))  # E2: written-off revenue as a share of revenue
    .withColumn("e3_grant_dependence", F.when(F.col("usable"), ratio("operational_grants", "revenue")))  # E3: operating grants as a share of revenue
    # F: spending structure
    .withColumn("f1_employee_cost_pct", F.when(F.col("usable"), ratio("employee_costs", "expenditure")))  # F1: salary bill as a share of spending
    .withColumn("f2_contracted_services_pct", F.when(F.col("usable"), ratio("contracted_services", "expenditure")))  # F2: contractors as a share of spending
    # G: context
    .withColumn("g1_category", F.col("category"))  # G1: A metro, B local, C district
    .withColumn("g2_province", F.col("province_code"))  # G2: province code
    .withColumn("g3_log_size", F.when(F.col("usable") & (F.col("expenditure") > 0), F.log10("expenditure"))))  # G3: size as log10 of total expenditure

FEATURES = ["a1_opinion_rank", "a2_non_reliable_now", "a3_non_reliable_last3", "a4_years_since_clean", "a5_ever_disclaimer", "a6_opinion_trend",  # the 25 features, in menu order
            "b1_missing_income_stmt", "b2_missing_balance_sheet", "b3_missing_debtors", "b4_fin_flagged_broken",
            "c1_irregular_pct", "c2_fruitless_pct", "c3_unauthorised_pct", "c4_irregular_trend",
            "d1_current_ratio", "d2_cash_coverage_months", "d3_operating_margin",
            "e1_debtors_over_90_pct", "e2_debt_impairment_pct", "e3_grant_dependence",
            "f1_employee_cost_pct", "f2_contracted_services_pct",
            "g1_category", "g2_province", "g3_log_size"]
out = feat.select(*KEY, "name", "opinion_code", "label", "fin_usable", *FEATURES)  # keep the identifiers, the label and the 25 features
out.write.mode("overwrite").format("delta").saveAsTable("features_muni_year")  # save as a Delta table: the model's training table
print(f"features_muni_year: {out.count():,} rows, {len(FEATURES)} features")  # report its size

## 4. Coverage: how much of each feature is filled in, in the modelling years 2019 to 2023

In [ ]:
f = spark.table("features_muni_year").where("year BETWEEN 2019 AND 2023 AND label IS NOT NULL")  # labelled rows in the likely modelling window
n = f.count()  # number of those rows
cov = f.select(*[(F.count(c) / n * 100).alias(c) for c in FEATURES]).first().asDict()  # for each feature, the % of rows that have a value
for c in FEATURES:  # go through each feature
    print(f"{c:30s} {cov[c]:5.1f}% filled")  # print its coverage

## 5. Checks

In [ ]:
def check(name, ok, detail=""):
    """Print PASS or FAIL for one check, with optional detail in brackets."""
    print(("PASS " if ok else "FAIL ") + name + (f"  ({detail})" if detail else ""))  # e.g. "PASS one row per municipality-year"

t = spark.table("features_muni_year")  # open the features table
check("one row per municipality-year", t.count() == panel.count(), f"{t.count()} vs {panel.count()}")  # check 1: no rows lost or duplicated by the joins
labelled = t.where("label IS NOT NULL").count()  # rows that have a next-year label
check("3,396 rows have a next-year label", labelled == 3396, labelled)  # check 2: matches the count worked out from the source
w = t.where("year BETWEEN 2019 AND 2023 AND label IS NOT NULL")  # the modelling window
pos = w.where("label = 1").count()  # non-reliable next year
check("2019-2023: 1,285 labelled rows, 173 non-reliable", (w.count(), pos) == (1285, 173), f"{w.count()} rows, {pos} non-reliable")  # check 3: matches decision 12's numbers
leak = t.where("year = 2024 AND label IS NOT NULL").count()  # 2024 has no 2025 opinion yet, so it must have no label
check("no label for the latest year (no peeking into the future)", leak == 0, leak)  # check 4
bad_fin = t.where("NOT coalesce(fin_usable, false) AND d1_current_ratio IS NOT NULL").count()  # financial features filled where they shouldn't be
check("financial features only where fin_usable (decisions 5 and 10)", bad_fin == 0, bad_fin)  # check 5
check("25 features", len(FEATURES) == 25, len(FEATURES))  # check 6